### Hyper-Parameter Tuning with CV

This notebook will cover exercise answer.

* Exercise 9.1
* Exercise 9.2
* Exercise 9.3
* Exercise 9.4
* Exercise 9.5
* Exercise 9.6

As we go along, there will be some explanations.

Most of the functions below can be found under Tool/metrics

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cqrlib as rs

%matplotlib inline

In [ ]:
X, y = rs.make_data(n_features=10, n_informative=5, n_redundant= 0, n_samples= 2000)

#Take note of depreciation warning Pandas 1.0.3

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC

n_splits = 10

param_grid = {'kernel':['rbf'], #redundant; default is already 'rbf'
              'C':[1e-2, 1e-1, 1, 10, 100],
              'gamma': [1e-2, 1e-1, 1, 10, 100]}

clf_svm = SVC(kernel='rbf',
              probability=True)

clf = GridSearchCV(estimator = clf_svm, 
                   param_grid = param_grid,
                   scoring='neg_log_loss', 
                   n_jobs=None, 
                   #iid='deprecated', take note of depreciation! version 0.22 warning
                   refit=True,
                   cv= n_splits, # 10 fold CV, but! stratified KFold
                   verbose=10)

clf.fit(X,
        y['bin'],
       sample_weight = None) #sample_weight here!!



**Note 1**

I allowed verbose = 10, so that we can see the output messages. Based on my due diligence and educated guess from these code snippets in the book, I believe Dr Marco was running sklearn version 0.14 - 0.16.

The only thing I found online for sklearn node_count was for decision tree, hence the answer provided is experimental (I think that was what Dr Marco wanted?).

[sklearn SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html?highlight=svc#sklearn.svm.SVC)

[sklearn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html?highlight=gridsearchcv#sklearn.model_selection.GridSearchCV)

[sklearn hyper-parameter tuning](https://scikit-learn.org/stable/modules/grid_search.html#gridsearch-scoring)

The upside of using sklearn 0.23.1, most of the function found in the book can be found in their current library.

* uniform log
* sample_weight can be passed to fit method

**Note 2**

As for how I derived combination of param_grids as "nodes" in the below answers. Kindly refer to [ML Mastery](https://machinelearningmastery.com/how-to-configure-the-number-of-layers-and-nodes-in-a-neural-network/#:~:text=A%20node%2C%20also%20called%20a,layers%20to%20comprise%20a%20network.)

The below is a diagram which I found from google.

![logo](https://pathmind.com/images/wiki/perceptron_node.png)

Although this is under Deep-ML, but given the time of writing we will have to perform an educated guess to match the author's intention as well as terms mentioned in the book (Match functionality to form connections).

Essentially, nodes are perceptors (Deep-Learning) which are **inputs**. In our case, Grid Search (GS) inputs are param_grids.

Hence, I believe this assumption does make sense in this GS exercise's context.

**Note 3**

At the same time, The results I placed below is mostly average/ mean. The reality is after splitting. Each node/ candidate were ran once on each sample.

In this example, I only generated 2,000 samples.

In [ ]:
def print_result(clf, n_splits):
    num_nodes = len(clf.cv_results_['rank_test_score'])
    total_time = clf.refit_time_ * num_nodes * n_splits

    best_scores, best_scores_idx, mean_score_time, mean_fit_time = [], 0, .0, .0

    for i in np.arange(n_splits):
        best_scores.append(clf.cv_results_["split"+ str(i) +"_test_score"][clf.best_index_])
        idx = np.where(max(best_scores) == best_scores)[0][0] #always + 1 because index starts from 0 as default

    best_scores_idx = (clf.best_index_ + 1) + len(clf.cv_results_['mean_score_time']) * idx
    print("Best params for estimator: {0}\nBest CV Score: {1:.6f}\n".format(clf.best_estimator_, max(best_scores)))
    print("A total of {0} was performed before optimal CV score found! (Under split{1}_test_score / {2}th split)\n".format(best_scores_idx, idx, idx + 1))

    print("Estimated time taken for entire process: {0:.6f} seconds\nTotal number of candidates/ nodes: {1}\n".format(total_time, 
                                                                                                           num_nodes))
    i = 0
    while i < (clf.best_index_ + 1):
        mean_score_time += clf.cv_results_['mean_score_time'][i]
        mean_fit_time += clf.cv_results_['mean_fit_time'][i]
        i+=1
    print("=" * 55)
    print("Estimated total mean time required for optimal solution:\n\nScore Time: {0:.6f}s\nFit Time: {1:.6f}s".format(mean_score_time, mean_fit_time))

print_result(clf = clf, n_splits = n_splits)

In [ ]:
from sklearn.utils.fixes import loguniform

param_distributions = {'C':loguniform(a = 1e-2, b= 1e2),
                      'gamma':loguniform(a = 1e-2, b= 1e2)}

Rclf = RandomizedSearchCV(estimator = clf_svm, 
                          param_distributions = param_distributions,
                          n_iter = 25, # newly added as well
                          scoring='neg_log_loss', 
                          n_jobs=None, 
                          #iid='deprecated', take note of depreciation! version 0.22 warning
                          refit=True,
                          cv= n_splits, # 10 fold CV, but! stratified KFold
                          verbose=10)

Rclf.fit(X,
         y['bin'],
         sample_weight = None) #sample_weight here!!

In [ ]:
print_result(clf = Rclf, n_splits = n_splits)

### Comparison between GSCV vs RSCV:

**Grid Search CV**

Best params for estimator: SVC(C=1, gamma=0.1, probability=True)

Best CV Score: -0.095049

A total of 87 was performed before optimal CV score found! (Under split3_test_score / 4th split)

Estimated time taken for entire process: 77.301800 seconds
Total number of candidates/ nodes: 25

Estimated total mean time required for optimal solution:

Score Time: 0.156009s
Fit Time: 6.964366s

**Random Search CV**

Best params for estimator: SVC(C=5.146914740190327, gamma=0.03550270078366642, probability=True)

Best CV Score: -0.100946 (Lowest)

A total of 78 was performed before optimal CV score found! (Under split3_test_score / 4th split)

Estimated time taken for entire process: 62.176406 seconds
Total number of candidates/ nodes: 25

Estimated total mean time required for optimal solution:

Score Time: 0.021389s
Fit Time: 1.155873s

### Conclusion

Number of nodes are both 25 (RSCV: n_iter = 25), (GSCV = 5 * 5 inputs).

While RSCV does perform significantly much faster for both score and fit time as well as the number of fits required.

GSCV has a higher neg_log_loss score (Not optimal prediction score as compared to RSCV).

Hence RSCV does seem overall better with a lower score and faster results.

But as seen in previous chapters, randomness will usually cause an inflated score (more bias). Hence, GSCV may provide a more reliable outcome.

**Note**

For sklearn, their attributes after fit will always return the highest mean score (Worst score for neg_log_loss). Hence the above comparison is more like comparing between the worst log_loss reported, which method provides the better prediction score (lowest).

In [ ]:
def in_sample_sharpe_ratio(clf):
    sharpe_ratio = []
    for i in np.arange(len(clf.cv_results_['mean_test_score'])):
        if clf.cv_results_['mean_test_score'][i] < 0:
            sharpe_ratio.append(-1 * clf.cv_results_['mean_test_score'][i]/ clf.cv_results_['std_test_score'][i])
        else:
            sharpe_ratio.append(clf.cv_results_['mean_test_score'][i]/ clf.cv_results_['std_test_score'][i])
    print("IS Best Score Sharpe Ratio: {0:.6f}".format(sharpe_ratio[clf.best_index_]))
    print("Best IS Sharpe ratio: {0:.6f}\nLowest IS Sharpe Ratio: {1:.6f}\nMean Sharpe Ratio: {2:.6f}".format(max(sharpe_ratio), min(sharpe_ratio), np.mean(sharpe_ratio)))

In [ ]:
in_sample_sharpe_ratio(clf)

In [ ]:
clf1 = GridSearchCV(estimator = clf_svm, 
                   param_grid = param_grid,
                   scoring='accuracy', #change to accuracy
                   n_jobs=None, 
                   #iid='deprecated', take note of depreciation! version 0.22 warning
                   refit=True,
                   cv= n_splits, # 10 fold CV, but! stratified KFold
                   verbose=10)

clf1.fit(X,
        y['bin'],
       sample_weight = None) #sample_weight here!!

in_sample_sharpe_ratio(clf1)

In [ ]:
in_sample_sharpe_ratio(Rclf)

In [ ]:
Rclf1 = RandomizedSearchCV(estimator = clf_svm, 
                          param_distributions = param_distributions,
                          n_iter = 25, # newly added as well
                          scoring='accuracy', 
                          n_jobs=None, 
                          #iid='deprecated', take note of depreciation! version 0.22 warning
                          refit=True,
                          cv= n_splits, # 10 fold CV, but! stratified KFold
                          verbose=10)

Rclf1.fit(X,
         y['bin'],
         sample_weight = None) #sample_weight here!!

in_sample_sharpe_ratio(Rclf1)

### Scoring Methods Comparison

To ensure fairness in comparing across both methods and scoring type. We will only take the "best" score given provided and lowest score that we can find.

When using RSCV, remember to provide random_state. Otherwise, your answers will keep changing.

GSCV (Neg_log_loss):

In-Sample Best Score Sharpe Ratio: 2.487544 (Lowest)

Lowest In-Sample Sharpe Ratio: 2.194502

GSCV (Accuracy):

In-Sample Best Score Sharpe Ratio: 21.373090

Lowest In-Sample Sharpe Ratio: 8.064150

RSCV (Neg_log_loss):

In-Sample Best Score Sharpe Ratio: 2.509210

Lowest In-Sample Sharpe Ratio: 2.175036 (Lowest)

RSCV (Accuracy):

In-Sample Best Score Sharpe Ratio: 23.442070

Lowest In-Sample Sharpe Ratio: 6.252751

**At-a-glance**

GridSearchCV (Neg_log_loss) produces a lowest sharpe ratio based on the best score provided.

However, when comparing with RSCV (Neg_log_loss) lowest in-sample sharpe ratio. It seems higher.

This demostrates a consistence in sharpe ratio.

This characteristic is carried over when using "accuracy" as a scoring method (RSCV has a lower IS SR: 6.252751).

When comparing between "accuracy" and "neg_log_loss", it is very obvious accuracy provides a very high sharpe ratio.

But it is even more obvious "accuracy" has a large difference between best and lowest SR.

It seems that "accuracy" as a scoring method is not very accurate.

This is also mentioned in AFML page 134, section 9.4.

### Conclusion

In short, strongly encourage cross-entropy (neg_log_loss) as a scoring method.

Refer to the below negative log loss formula (Binary classes):

![logo](http://wiki.fast.ai/images/2/28/Cross_entropy_formula.png)
    
For natural logarithm if p was to be < 1. The answer would always be negative. Hence to make it let less confusing and comparable to other metrics, we adopt negative sign (Minimize negative log loss).

If you maximise log loss instead (Remove negative sign only/ Maximum log likelihood), your probability would be opposite of the below. Your prediction probability will keep increasing rapidly (Instead of penalizing errorrous prediction, it will end up endorsing such mistakes).

![log_loss](http://wiki.fast.ai/images/4/43/Log_loss_graph.png)

When you have a strategy that bets equally. Accuracy may prove to be bad choice since they attribute true prediction with erroreous prediction equally (your predicted value equals the actual value). 

Under Kelly's criterion, if your lose rate is 50/50 it will lead to a quick ruination.(Recall 2p-1)

If sample weights were included, accuracy may be considered. In most cases cross-entropy is a better choice, since it will factor in the uncertainty based on how much it varies from actual label.

Update: Up till this stage, I realised Dr Marco teaches deep technical knowledge with useful resources to develop further understanding. But he only ask key conceptual questions.